# DCP for time series: decisions and benchmarks

DCP with panel quantile regression, a Newsvendor solver over the calibrated distribution, and economic benchmarks with tail-risk metrics.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from mlforecast import MLForecast
from sklearn.ensemble import HistGradientBoostingRegressor

from tinyconformal.series import DistributionalConformalPredictiveSystemTimeSeriesRegressor
from tinyconformal.utils import NewsvendorSolver

## Demand panel

Four heterogeneous monthly series, with the final 12 months reserved for out-of-sample benchmarking.

In [ ]:
rng = np.random.default_rng(42)
n_series, periods, horizon = 4, 96, 12
dates = pd.date_range("2018-01-01", periods=periods, freq="MS")
rows = []
for item in range(n_series):
    t = np.arange(periods)
    location = 30 + 6 * item + 0.15 * t + 9 * np.sin(2 * np.pi * t / 12 + item / 4)
    demand = np.maximum(location + rng.normal(0, 3 + item, periods), 0)
    rows.append(pd.DataFrame({"unique_id": f"sku_{item}", "ds": dates, "y": demand}))
df = pd.concat(rows, ignore_index=True)
train = df.groupby("unique_id", group_keys=False).head(periods - horizon).reset_index(drop=True)
test = df.groupby("unique_id", group_keys=False).tail(horizon).reset_index(drop=True)

## Quantile grid and DCP

DCP conformalizes the base CDF using horizon-specific backtesting PITs. A relatively dense grid makes it possible to query the economic fractile without limiting the benchmark to a single interval.

In [ ]:
levels = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
quantile_columns = {level: f"HGBR-q-{level * 100:g}" for level in levels}
models = {
    quantile_columns[level]: HistGradientBoostingRegressor(
        loss="quantile", quantile=level, max_iter=150, max_leaf_nodes=15, random_state=42
    )
    for level in levels
}
learner = MLForecast(models=models, freq="MS", lags=[1, 2, 3, 6, 12])
dcp = DistributionalConformalPredictiveSystemTimeSeriesRegressor(
    learner=learner, horizon=horizon, quantile_columns=quantile_columns, n_windows=4, alpha=0.10
).fit(train, static_features=[], n_jobs=1)
forecast, distributions = dcp.predict_distribution(h=horizon)
distribution = distributions["HGBR"]
forecast[["q10_dcp", "q50_dcp", "q90_dcp"]] = distribution.ppf(
    np.broadcast_to([0.10, 0.50, 0.90], (len(forecast), 3))
)
forecast.head()

In [ ]:
dcp.evaluate(test, h=horizon)

## Solver Newsvendor

The solver converts costs into a critical fractile and evaluates the DCP PPF directly. Here, $c_u=8$ and $c_o=2$, so $q^*=0.8$.

In [ ]:
decision = forecast.copy()
decision["underage_cost"] = 8.0
decision["overage_cost"] = 2.0
decision = NewsvendorSolver.optimize_distribution(
    decision, distribution, underage_cost="underage_cost", overage_cost="overage_cost"
)
decision["Optimal DCP"] = decision.pop("y_optimal")
decision["DCP median"] = decision["q50_dcp"]
decision["Base 75% quantile"] = decision[quantile_columns[0.75]]
decision["Seasonal naive"] = train.groupby("unique_id")["y"].tail(horizon).to_numpy()
decision["y"] = test["y"].to_numpy()
decision[["unique_id", "ds", "critical_ratio", "Optimal DCP", "DCP median", "Base 75% quantile", "Seasonal naive", "y"]].head()

## Economic loss and tail risk

The benchmark uses exactly the same cost function optimized by the solver. The 95% VaR/CVaR metrics highlight models with rare but financially severe losses.

In [ ]:
models_benchmark = ["Optimal DCP", "DCP median", "Base 75% quantile", "Seasonal naive"]

def period_loss(frame, model):
    actual = frame["y"].to_numpy(float)
    order = frame[model].to_numpy(float)
    cu = frame["underage_cost"].to_numpy(float)
    co = frame["overage_cost"].to_numpy(float)
    return cu * np.maximum(actual - order, 0) + co * np.maximum(order - actual, 0)

def economic_loss(frame, model_names):
    return pd.DataFrame({"model": model_names, "economic_loss": [period_loss(frame, m).sum() for m in model_names]}).sort_values("economic_loss")

def tail_risk(frame, model_names, risk_level=0.95):
    rows = []
    for model in model_names:
        loss = period_loss(frame, model)
        var = np.quantile(loss, risk_level)
        tail = loss[loss >= var]
        rows.append({"model": model, "expected_loss": loss.mean(), "VaR_95": var, "CVaR_95": tail.mean(), "worst_loss": loss.max()})
    return pd.DataFrame(rows).sort_values("CVaR_95")

In [ ]:
economic_loss(decision, models_benchmark)

In [ ]:
tail_risk(decision, models_benchmark)